<a href="https://colab.research.google.com/github/Brian342/Chronic-Disease-Risk/blob/main/cervical_cancer_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install packages

In [ ]:
# Connect the drive to colab
from google.colab import drive
drive.mount('/content/drive',force_remount=True)


Mounted at /content/drive


In [ ]:
# ============================================================
# STEP 2: Unzip your dataset
# Replace the path below with the actual path to your zip file
# ============================================================
import zipfile
import os

# --- UPDATE THIS PATH to your zip file location in Drive ---
zip_path = '/content/drive/MyDrive/cervical2.zip'   # <-- change this

# Where to extract the dataset inside Colab's runtime
extract_to = '/content/dataset'

os.makedirs(extract_to, exist_ok=True)

print("Unzipping dataset...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)
print(f"Done! Dataset extracted to: {extract_to}")

# ============================================================
# STEP 3: Verify the folder structure
# ============================================================
for root, dirs, files in os.walk(extract_to):
    level = root.replace(extract_to, '').count(os.sep)
    indent = '    ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:   # Only show files 2 levels deep to keep output clean
        sub_indent = '    ' * (level + 1)
        for f in files[:3]:   # Show first 3 files per folder as a sample
            print(f"{sub_indent}{f}")
        if len(files) > 3:
            print(f"{sub_indent}... and {len(files) - 3} more files")

# ============================================================
# STEP 4: Quick dataset summary
# ============================================================
print("\n--- Dataset Summary ---")
for split in ['training', 'testing']:
    split_path = os.path.join(extract_to, split)
    if os.path.exists(split_path):
        print(f"\n{split.upper()} SET:")
        for class_name in sorted(os.listdir(split_path)):
            class_path = os.path.join(split_path, class_name)
            if os.path.isdir(class_path):
                count = len(os.listdir(class_path))
                print(f"  {class_name:<30} {count} images")

Unzipping dataset...
Done! Dataset extracted to: /content/dataset
dataset/
    Herlev Dataset/
        train/
            normal_columnar/
            moderate_dysplastic/
            light_dysplastic/
            normal_superficiel/
            severe_dysplastic/
            carcinoma_in_situ/
            normal_intermediate/
        test/
            normal_columnar/
            moderate_dysplastic/
            light_dysplastic/
            normal_superficiel/
            severe_dysplastic/
            carcinoma_in_situ/
            normal_intermediate/

--- Dataset Summary ---


In [ ]:
# =======================
# Cell 1 - Imports & config
# =======================
import os
from pathlib import Path
import shutil
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import re
from collections import Counter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications.resnet import preprocess_input, ResNet152

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
def load_sipakmed(root: Path, mapping: dict, img_exts={".jpg",".jpeg",".png",".tif",".tiff",".bmp"}):
    records = []
    for class_dir in root.iterdir():
        if not class_dir.is_dir():
            continue
        class_name = class_dir.name
        if class_name not in mapping:
            continue
        label = mapping[class_name]

        # 🔹 Search two levels deep (handles nested + CROPPED)
        for img in class_dir.rglob("*"):
            if img.suffix.lower() in img_exts:
                records.append((str(img), label))
    return records


In [ ]:
sipakmed_map = {
    "im_Parabasal": "abnormal",
    "im_Dyskeratotic": "abnormal",
    "im_Metaplastic": "abnormal",
    "im_Koilocytotic": "abnormal",
    "im_Superficial-Intermediate": "normal"
}

sipakmed_records = load_sipakmed(Path("/kaggle/input/sipakmed"), sipakmed_map)
df_sipakmed = pd.DataFrame(sipakmed_records, columns=["path","class"])
df_sipakmed["label"] = df_sipakmed["class"].map({"normal":0,"abnormal":1})


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/sipakmed'

In [ ]:
from pathlib import Path
import pandas as pd

img_exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

def load_dataset_with_mapping(root: Path, mapping: dict, recursive=True):
    records = []
    for class_dir in root.iterdir():
        if not class_dir.is_dir():
            continue
        class_name = class_dir.name
        if class_name not in mapping:
            continue
        label = mapping[class_name]

        # Recurse for subfolders (handles SipakMed CROPPED)
        search = class_dir.rglob("*") if recursive else class_dir.glob("*")
        for img in search:
            if img.suffix.lower() in img_exts:
                records.append((str(img), label))
    return records


In [ ]:
mendeley_map = {
    "Negative for Intraepithelial malignancy": "normal",
    "Low squamous intra-epithelial lesion": "abnormal",
    "High squamous intra-epithelial lesion": "abnormal",
    "Squamous cell carcinoma": "abnormal"
}

mendeley_records = load_dataset_with_mapping(
    Path("/kaggle/input/mendeley-lbc-cervical-cancer"), mendeley_map, recursive=True
)
df_mendeley = pd.DataFrame(mendeley_records, columns=["path","class"])
df_mendeley["label"] = df_mendeley["class"].map({"normal":0,"abnormal":1})


In [ ]:
herlev_map = {
    "normal_superficiel": "normal",
    "normal_intermediate": "normal",
    "light_dysplastic": "abnormal",
    "normal_columnar": "normal",
    "moderate_dysplastic": "abnormal",
    "severe_dysplastic": "abnormal",
    "carcinoma_in_situ": "abnormal"
}

herlev_root = Path("/kaggle/input/herlev-dataset/Herlev Dataset")

herlev_records = []
for split in ["train", "test"]:
    split_path = herlev_root / split
    herlev_records += load_dataset_with_mapping(split_path, herlev_map, recursive=True)

df_herlev = pd.DataFrame(herlev_records, columns=["path","class"])
df_herlev["label"] = df_herlev["class"].map({"normal":0,"abnormal":1})


In [ ]:
print("SipakMed:", df_sipakmed["class"].value_counts())
print("Herlev:", df_herlev["class"].value_counts())
print("Mendeley:", df_mendeley["class"].value_counts())


In [ ]:
from sklearn.model_selection import train_test_split

def split_dataset(df, test_size=0.2, seed=42):
    return train_test_split(
        df["path"], df["label"],
        test_size=test_size, stratify=df["label"], random_state=seed
    )

In [ ]:
X1_train, X1_test, y1_train, y1_test = split_dataset(df_sipakmed)
X2_train, X2_test, y2_train, y2_test = split_dataset(df_herlev)
X3_train, X3_test, y3_train, y3_test = split_dataset(df_mendeley)

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet152
from tensorflow.keras.applications.resnet import preprocess_input
from tensorflow.keras.preprocessing import image
import numpy as np

# Load pretrained ResNet152 (without top FC layers)
resnet = ResNet152(weights="imagenet", include_top=False, pooling="avg")

def extract_features(paths, target_size=(224,224), batch_size=32):
    features = []
    for i in range(0, len(paths), batch_size):
        batch_paths = paths[i:i+batch_size]
        imgs = []
        for p in batch_paths:
            img = image.load_img(p, target_size=target_size)
            arr = image.img_to_array(img)
            arr = np.expand_dims(arr, axis=0)
            imgs.append(arr)
        batch = np.vstack(imgs)
        batch = preprocess_input(batch)
        feats = resnet.predict(batch, verbose=0)
        features.append(feats)
    return np.vstack(features)


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import seaborn as sns

In [ ]:
resnet = ResNet152(weights='imagenet', include_top=False, pooling='avg')
print("ResNet152 loaded. Output feature dim:", resnet.output_shape)

In [ ]:

def split_dataset(df, test_size=0.2, seed=SEED):
    return train_test_split(df['path'], df['label'], test_size=test_size, stratify=df['label'], random_state=seed)


def extract_features(paths, dataset_name, split_name, target_size=(224,224), batch_size=32, cache_dir='features_cache'):
    """Extract features using ResNet152 and cache to disk per dataset/split.
    paths: list-like of image paths
    dataset_name: string (used for cache filename)
    split_name: 'train' or 'test'
    Returns: numpy array of shape (N, feature_dim)
    """
    os.makedirs(cache_dir, exist_ok=True)
    cache_path = os.path.join(cache_dir, f"{dataset_name}_{split_name}_resnet152.npy")

    if os.path.exists(cache_path):
        print(f"Loading cached features: {cache_path}")
        return np.load(cache_path)

    n = len(paths)
    if n == 0:
        return np.zeros((0, resnet.output_shape[-1]))

    feats_list = []
    for i in tqdm(range(0, n, batch_size), desc=f"Extract {dataset_name}:{split_name}"):
        batch_paths = paths[i:i+batch_size]
        imgs = []
        for p in batch_paths:
            try:
                img = image.load_img(p, target_size=target_size)
                arr = image.img_to_array(img)
            except Exception as e:
                print(f"Warning: failed to load {p}: {e}")
                arr = np.zeros((target_size[0], target_size[1], 3), dtype=np.float32)
            imgs.append(arr)

        batch = np.stack(imgs, axis=0)
        batch = preprocess_input(batch)
        feats = resnet.predict(batch, verbose=0)
        feats_list.append(feats)

    feats_all = np.vstack(feats_list)
    np.save(cache_path, feats_all)
    print(f"Saved features to {cache_path}")
    return feats_all

Mounted at /content/drive


In [ ]:
def train_and_evaluate(df, dataset_name, test_size=0.2, cache_dir='features_cache', model_dir='models'):
    """
    Splits dataset, extracts ResNet152 features, trains Logistic Regression (SLR),
    evaluates and plots confusion matrix, and saves features & model.

    Returns: dict with metrics and paths to artifacts.
    """
    import os
    import joblib
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.preprocessing import LabelEncoder
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
    from tqdm import tqdm

    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(cache_dir, exist_ok=True)

    print(f"\n📊 Training on {dataset_name}")

    # 1️⃣ Split
    X_train, X_test, y_train, y_test = train_test_split(
        df['path'], df['label'], test_size=test_size,
        stratify=df['label'], random_state=SEED
    )

    # 2️⃣ Encode labels as strings (important!)
    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train.astype(str))
    y_test_enc  = le.transform(y_test.astype(str))

    # 3️⃣ Extract features
    def extract_features(paths, split_name):
        cache_path = os.path.join(cache_dir, f"{dataset_name}_{split_name}_resnet152.npy")
        if os.path.exists(cache_path):
            return np.load(cache_path)

        feats_list = []
        batch_size = 32
        for i in tqdm(range(0, len(paths), batch_size), desc=f"Extract {dataset_name}:{split_name}"):
            batch_paths = paths[i:i+batch_size]
            imgs = []
            for p in batch_paths:
                try:
                    img = tf.keras.preprocessing.image.load_img(p, target_size=(224,224))
                    arr = tf.keras.preprocessing.image.img_to_array(img)
                except:
                    arr = np.zeros((224,224,3), dtype=np.float32)
                imgs.append(arr)
            batch = np.stack(imgs, axis=0)
            batch = preprocess_input(batch)
            feats = resnet.predict(batch, verbose=0)
            feats_list.append(feats)
        feats_all = np.vstack(feats_list)
        np.save(cache_path, feats_all)
        return feats_all

    train_feats = extract_features(list(X_train), 'train')
    test_feats  = extract_features(list(X_test), 'test')

    print(f"Train features: {train_feats.shape}, Test features: {test_feats.shape}")

    # 4️⃣ Train Logistic Regression
    clf = LogisticRegression(max_iter=2000, solver='saga', penalty='l2', random_state=SEED)
    clf.fit(train_feats, y_train_enc)

    # Save model & label encoder
    model_path = os.path.join(model_dir, f"{dataset_name}_logreg.joblib")
    joblib.dump({'clf': clf, 'label_encoder': le}, model_path)
    print(f"Saved classifier to {model_path}")

    # 5️⃣ Predictions & metrics
    y_pred = clf.predict(test_feats)

    # Classification report
    target_names = [str(c) for c in le.classes_]  # <-- convert to strings
    report = classification_report(y_test_enc, y_pred, target_names=target_names, output_dict=True)
    print("\nClassification Report:\n", classification_report(y_test_enc, y_pred, target_names=target_names))

    # ROC-AUC (use predicted probabilities if available)
    try:
        y_prob = clf.predict_proba(test_feats)[:,1]
        roc = roc_auc_score(y_test_enc, y_prob)
    except Exception:
        roc = roc_auc_score(y_test_enc, y_pred)
    print("ROC-AUC:", roc)

    # Confusion matrix
    cm = confusion_matrix(y_test_enc, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_names, yticklabels=target_names)
    plt.title(f"Confusion Matrix - {dataset_name}")
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

    return {
        'dataset': dataset_name,
        'model_path': model_path,
        'feature_cache_train': os.path.join(cache_dir, f"{dataset_name}_train_resnet152.npy"),
        'feature_cache_test': os.path.join(cache_dir, f"{dataset_name}_test_resnet152.npy"),
        'report': report,
        'roc_auc': roc,
        'confusion_matrix': cm
    }


In [ ]:



























































































































































































































































































































































































































































































































































esults = {}

datasets = [
    (globals().get('df_sipakmed'), 'SipakMed'),
    (globals().get('df_herlev'), 'Herlev'),
    (globals().get('df_mendeley'), 'Mendeley_LBC')
]

for df, name in datasets:
    if df is None:
        print(f"DataFrame for {name} not found in workspace. Skipping.")
        continue
    results[name] = train_and_evaluate(df, name)

# ==============================
# Summary table of metrics
# ==============================
summary = []

for name, res in results.items():
    r = res['report']
    acc = r.get('accuracy', None)
    f_normal = r.get('normal', {}).get('f1-score', None)
    f_abnormal = r.get('abnormal', {}).get('f1-score', None)
    roc_auc = res['roc_auc']

    summary.append({
        'dataset': name,
        'accuracy': acc,
        'f1_normal': f_normal,
        'f1_abnormal': f_abnormal,
        'roc_auc': roc_auc
    })

if summary:
    df_summary = pd.DataFrame(summary)
    display(df_summary)


DataFrame for SipakMed not found in workspace. Skipping.
DataFrame for Herlev not found in workspace. Skipping.
DataFrame for Mendeley_LBC not found in workspace. Skipping.


NameError: name 'results' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')